# 🎬 JUDGEMENT TALES — free open-source animation (Colab)
**WANNATION MEDIA.** This notebook turns your scene art into an animated, voiced, subtitled clip —
100% open-source models, running on Google's **free GPU**. No subscriptions, no sign-ups, no paid API.

**Stack:** LTX-Video (motion) · Piper (voices) · FFmpeg (assembly) — all free & open.

### How to run
1. Top menu → **Runtime → Change runtime type → T4 GPU** (free) → Save.
2. Run each cell top to bottom (▶). When asked, **upload your scene images**.
3. Last cell downloads `judgement_tales_clip.mp4`.

> Honest note: LTX gives real **scene/camera motion** of your art. Full lip-synced talking mouths are an
> optional extra stage (SadTalker, noted at the bottom). This notebook gives you a genuine animated, voiced,
> watermarked clip of your scenes — the real first step of the series, made for free.

## 0 · Check you have the free GPU

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout or 'no nvidia-smi')
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then re-run.'
print('GPU ready:', torch.cuda.get_device_name(0))

## 1 · Install the open-source tools (~2–3 min, first run only)

In [ ]:
%pip -q install "diffusers>=0.32" transformers accelerate imageio imageio-ffmpeg sentencepiece
%pip -q install piper-tts onnxruntime
import imageio_ffmpeg, os
os.environ['FFMPEG'] = imageio_ffmpeg.get_ffmpeg_exe()
print('ffmpeg:', os.environ['FFMPEG'])
print('installed.')

## 2 · Upload your scene images
Upload the JUDGEMENT TALES scene frames (the 5 stills, or any in-style art you make). JPG/PNG.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('images', exist_ok=True)
up = files.upload()  # pick your scene images
for name in up:
    shutil.move(name, os.path.join('images', name))
imgs = sorted(os.path.join('images', f) for f in os.listdir('images') if f.lower().endswith(('.jpg','.jpeg','.png')))
print('uploaded:', imgs)

## 3 · The shot list (edit freely)
Each shot = one image + a motion prompt + a character voice + a spoken line.
Change the filenames to match what you uploaded, rewrite lines, add shots.

In [ ]:
# (image path, motion prompt for LTX, voice key, spoken line)
SHOTS = [
    ('images/1.jpg', 'subtle camera push-in, people shifting in a queue, gentle ambient motion, cartoon', 'peter',  'Take a ticket. Zero four seven.'),
    ('images/1.jpg', 'slight head turn and hand movement, cartoon character reacting', 'lwazi',  'Sharp. Okay. I am not even mad.'),
    ('images/4.jpg', 'robes swaying, a grand gesture of the hand, cartoon', 'sultan', 'The Sultan does not queue. The Sultan visits.'),
    ('images/3.jpg', 'papers fluttering on a desk, a weary sigh, cartoon', 'moses',  'Complaint number one. Water. I have heard this one before.'),
    ('images/2.jpg', 'presenter gesturing at a glowing slide, cartoon', 'jesus',  'Great question. Let us park it.'),
    ('images/5.jpg', 'a boardroom pie chart shimmering, subtle nods, cartoon', 'analyst','Just rename Pending to In Transition.'),
]
# characters without a real SA-accent open voice fall back to the closest free Piper voice (see cell 5).
print(len(SHOTS), 'shots configured')

## 4 · Stage 2 — animate each still (LTX-Video, free & open)
Turns each image into a short motion clip. ~1–3 min per shot on the free T4.

In [ ]:
import torch, os, imageio
from PIL import Image
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

pipe = LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()

W, H, NFRAMES, FPS = 768, 512, 49, 12  # ~4s clips
os.makedirs('shots', exist_ok=True)

def cover(img, w, h):
    s = max(w/img.width, h/img.height)
    img = img.resize((round(img.width*s), round(img.height*s)))
    l=(img.width-w)//2; t=(img.height-h)//2
    return img.crop((l,t,l+w,t+h))

shot_paths = []
for i,(img_path, prompt, voice, line) in enumerate(SHOTS):
    print(f'[{i+1}/{len(SHOTS)}] animating {img_path} ...')
    image = cover(Image.open(img_path).convert('RGB'), W, H)
    out = pipe(image=image, prompt=prompt,
               negative_prompt='worst quality, blurry, distorted, warped faces',
               width=W, height=H, num_frames=NFRAMES, num_inference_steps=40,
               generator=torch.manual_seed(42+i)).frames[0]
    p = f'shots/shot{i:02d}.mp4'
    export_to_video(out, p, fps=FPS)
    shot_paths.append(p)
    print('   ->', p)
print('done:', shot_paths)

## 5 · Stage 1 — voices (Piper, free & offline)
Downloads a few open Piper voices and speaks each line. Piper has no South-African voice yet, so
characters use the closest free English voices as stand-ins. (Upgrade path: clone real SA voices with
XTTS from short recordings — noted at the bottom.)

In [ ]:
import os, urllib.request, subprocess, json, wave, contextlib
os.makedirs('voices', exist_ok=True); os.makedirs('audio', exist_ok=True)
BASE='https://huggingface.co/rhasspy/piper-voices/resolve/main/en'
VOICES = {
  'ryan':  f'{BASE}/en_US/ryan/medium/en_US-ryan-medium',
  'lessac':f'{BASE}/en_US/lessac/medium/en_US-lessac-medium',
  'amy':   f'{BASE}/en_US/amy/medium/en_US-amy-medium',
  'alan':  f'{BASE}/en_GB/alan/medium/en_GB-alan-medium',
}
for key,url in VOICES.items():
    for ext in ('.onnx','.onnx.json'):
        dst=f'voices/{key}{ext}'
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url+ext, dst)
# map characters -> a downloaded voice
CHAR_VOICE = {'lwazi':'ryan','peter':'alan','sultan':'lessac','moses':'alan','jesus':'ryan','analyst':'amy','brenda':'amy'}

def wav_dur(path):
    with contextlib.closing(wave.open(path,'r')) as w:
        return w.getnframes()/float(w.getframerate())

line_wavs=[]; durations=[]
for i,(img_path,prompt,voice,line) in enumerate(SHOTS):
    vk = CHAR_VOICE.get(voice,'ryan')
    out=f'audio/line{i:02d}.wav'
    subprocess.run(f'echo {json.dumps(line)} | piper -m voices/{vk}.onnx -f {out}', shell=True, check=True)
    line_wavs.append(out); durations.append(wav_dur(out))
    print(f'[{i+1}] {voice}->{vk}: {durations[-1]:.1f}s  "{line}"')
print('voiced', len(line_wavs), 'lines')

## 6 · Stage 4 — assemble (watermark + subtitles + voices) → final clip
Runs on CPU. Pads each shot to its voice line, stitches, burns English subtitles, stamps the slow-spinning
WANNATION watermark, and muxes the audio.

In [ ]:
import subprocess, os, math
from PIL import Image, ImageDraw, ImageFont

# --- spinning WANNATION badge (PIL, no ImageMagick needed) ---
logo=Image.new('RGBA',(500,500),(0,0,0,0)); d=ImageDraw.Draw(logo)
d.ellipse((30,30,470,470),fill=(18,16,28,255))
d.ellipse((30,30,470,470),outline=(232,194,74,255),width=16)
try:
    f=ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf',150)
except: f=ImageFont.load_default()
d.text((250,235),'WN',font=f,fill=(232,194,74,255),anchor='mm')
logo.save('logo.png')

FF=os.environ.get('FFMPEG','ffmpeg')
os.makedirs('seg', exist_ok=True)
def run(c): subprocess.run(c, shell=True, check=True)

# per shot: pad video to the voice length, attach the voice
segs=[]; srt=[]; t=0.0
for i,((img,prompt,voice,line),vid,wav,dur) in enumerate(zip(SHOTS,shot_paths,line_wavs,durations)):
    seg=f'seg/seg{i:02d}.mp4'; d=max(dur,2.0)+0.3
    run(f'{FF} -y -i {vid} -i {wav} -filter_complex "[0:v]tpad=stop_mode=clone:stop_duration=30,trim=0:{d:.2f},setpts=PTS-STARTPTS[v]" -map "[v]" -map 1:a -c:v libx264 -pix_fmt yuv420p -c:a aac -t {d:.2f} {seg}')
    segs.append(seg)
    def ts(x):
        h=int(x//3600); m=int((x%3600)//60); s=x%60; return f'{h:02d}:{m:02d}:{s:06.3f}'.replace('.',',')
    srt.append(f'{i+1}\n{ts(t)} --> {ts(t+d)}\n{line}\n')
    t+=d
open('subs.srt','w').write('\n'.join(srt))
open('list.txt','w').write('\n'.join(f"file '{s}'" for s in segs))

run(f'{FF} -y -f concat -safe 0 -i list.txt -c copy _base.mp4')
vf=("[1:v]scale=120:120,rotate=2*PI*t/25:c=none:ow=rotw(iw):oh=roth(ih)[wm];"
    "[0:v][wm]overlay=W-w-32:32:format=auto:shortest=1[o];"
    "[o]subtitles=subs.srt:force_style='FontName=DejaVu Serif,Fontsize=20,Outline=2,Shadow=1'[v]")
run(f'{FF} -y -i _base.mp4 -loop 1 -i logo.png -filter_complex "{vf}" -map "[v]" -map 0:a -c:v libx264 -crf 20 -pix_fmt yuv420p -c:a aac -shortest -movflags +faststart judgement_tales_clip.mp4')
print('DONE -> judgement_tales_clip.mp4')

## 7 · Watch & download

In [ ]:
from IPython.display import HTML; from base64 import b64encode
mp4=open('judgement_tales_clip.mp4','rb').read()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{b64encode(mp4).decode()}"></video>')

In [ ]:
from google.colab import files
files.download('judgement_tales_clip.mp4')

## 8 · Optional upgrades (all free & open)
- **More scenes (Stage 0):** generate extra in-style frames with **SDXL** (`diffusers`) or **FLUX.1-schnell** from the
  Episode 1 script, so you have one still per shot for a full episode. Train a small **LoRA** on your 5 frames to keep
  characters on-model.
- **Talking mouths (Stage 3):** run **SadTalker** or **LivePortrait** on a character's face + the voice wav to lip-sync.
- **Real SA / Mind-Your-Language voices:** clone them with **Coqui XTTS-v2** from ~10s reference recordings per character
  (replaces the Piper stand-ins). Fully local.
- **Longer episode:** add shots to `SHOTS`, re-run. Assembly scales automatically.

Ask Claude to wire any of these into this notebook.